# DMLS CA4 - Question 3: PyTorch profiling

This notebook loads the CA4 test data and the same feed-forward architecture used in Questions 1 and 2. It profiles 100 CPU batches, reports time and memory for Linear, BatchNorm, and activation modules, and compares ReLU, Sigmoid, Tanh, and GeLU. No model training is performed.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.profiler import ProfilerActivity, profile
from torch.utils.data import DataLoader

HERE = Path.cwd()
if not (HERE / 'ca4_common.py').is_file():
    raise FileNotFoundError('Run this notebook from the CA4 directory containing ca4_common.py')
sys.path.insert(0, str(HERE))

from ca4_common import FeedForwardClassifier, load_dataset, set_random_seed

In [ ]:
DATA_DIR = HERE / 'data'
BATCH_SIZE = 32
PROFILE_BATCHES = 100
ACTIVATIONS = ['relu', 'sigmoid', 'tanh', 'gelu']

set_random_seed(1404)
test_dataset = load_dataset(DATA_DIR, 'test')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'test samples: {len(test_dataset):,}; batches profiled per activation: {PROFILE_BATCHES}')

## Profiling helper

The model emits named `record_function` regions for each module. Self CPU time avoids double-counting nested operators. Positive self CPU memory is reported as allocator-visible memory attributed to each region; transient allocations freed during a region can make net memory zero or negative in lower-level operator rows.

In [ ]:
def profile_activation(activation: str, batches: int = PROFILE_BATCHES):
    set_random_seed(1404)
    model = FeedForwardClassifier(
        activation=activation,
        profile_modules=True,
    ).cpu().eval()

    # Warm-up keeps one-time dispatcher/setup work out of the measured region.
    warmup_features, _ = next(iter(test_loader))
    with torch.inference_mode():
        for _ in range(5):
            model(warmup_features.float())

    with profile(
        activities=[ProfilerActivity.CPU],
        profile_memory=True,
        record_shapes=True,
    ) as profiler:
        with torch.inference_mode():
            for batch_index, (features, _) in enumerate(test_loader):
                if batch_index >= batches:
                    break
                model(features.float())

    rows = []
    for event in profiler.key_averages():
        if event.key.startswith(('Linear::', 'BatchNorm::', 'Activation::')):
            rows.append({
                'activation': activation,
                'module': event.key,
                'calls': event.count,
                'self_cpu_time_ms': event.self_cpu_time_total / 1_000.0,
                'total_cpu_time_ms': event.cpu_time_total / 1_000.0,
                'self_cpu_memory_kib': event.self_cpu_memory_usage / 1024.0,
                'total_cpu_memory_kib': event.cpu_memory_usage / 1024.0,
            })
    return pd.DataFrame(rows), profiler


In [ ]:
profiles = {}
frames = []
for activation in ACTIVATIONS:
    frame, profiles[activation] = profile_activation(activation)
    frames.append(frame)

module_results = pd.concat(frames, ignore_index=True)
module_results.round(3)

## Question 3.1 - Linear, BatchNorm, and ReLU comparison

In [ ]:
relu_modules = module_results[module_results['activation'] == 'relu'].copy()
relu_modules[['module', 'calls', 'total_cpu_time_ms', 'total_cpu_memory_kib']].round(3)

In [ ]:
profiles['relu'].key_averages().table(
    sort_by='self_cpu_time_total', row_limit=20
)

## Question 3.2 - activation comparison

In [ ]:
activation_results = (
    module_results[module_results['module'].str.startswith('Activation::')]
    .groupby('activation', as_index=False)
    .agg(
        calls=('calls', 'sum'),
        total_cpu_time_ms=('total_cpu_time_ms', 'sum'),
        total_cpu_memory_kib=('total_cpu_memory_kib', 'sum'),
    )
)
activation_results.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
activation_results.plot.bar(
    x='activation', y='total_cpu_time_ms', legend=False, ax=axes[0], color='#2878B5'
)
axes[0].set_title('Activation CPU time over 100 batches')
axes[0].set_ylabel('Total CPU time (ms)')
axes[0].set_xlabel('')
activation_results.plot.bar(
    x='activation', y='total_cpu_memory_kib', legend=False, ax=axes[1], color='#E07A5F'
)
axes[1].set_title('Activation memory attributed by profiler')
axes[1].set_ylabel('CPU memory (KiB)')
axes[1].set_xlabel('')
plt.tight_layout();

## Report notes

Copy the two result tables and chart into the report. Discuss both absolute measurements and relative ordering. Profiler memory is operator-attributed allocation traffic, not whole-process peak RSS, so state that limitation explicitly. Results depend on Colab's assigned CPU and should be reported together with the runtime date and CPU model.